In [ ]:
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, str(Path.cwd().parent))

In [1]:
from paths import MASTER_PATH

In [16]:
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

In [ ]:
# The master panel with the first 5 rows
con.sql(f"""
    SELECT *
    FROM read_parquet('{MASTER_PATH}')
    LIMIT 5
""").df()

,trading_date,instrument_id,raw_symbol,instrument_class,strike_price,expiration_date,underlying_id,days_to_expiry,ttm_years,settlement_price,implied_vol,futures_price,moneyness_k_over_f
0,2024-01-15,107078955,TFO FMF0025_OMCE0000148002122724,C,148.0,2024-12-27,6884100,347,0.950685,1.913,0.9747,29.923,4.946028
1,2024-01-15,105285927,TFO FMG0025_OMCE0000031002012725,C,31.0,2025-01-27,6924878,378,1.035616,11.300,0.6599,29.923,1.035992
2,2024-01-25,105398527,TFO FMZ0024_OMCE0000044002112624,C,44.0,2024-11-26,6845512,306,0.838356,5.677,0.6765,27.784,1.583645
3,2024-02-13,107034754,TFO FMM0024_OMPE0000141002052424,P,141.0,2024-05-24,6687715,101,0.276712,114.902,1.0666,25.436,5.543324
4,2024-02-13,107083016,TFO FMH0025_OMPE0000146002022425,P,146.0,2025-02-24,6958602,377,1.032877,116.439,0.8898,25.436,5.739896


The master table contains $5615794$ records for $20252$.

The period from 2023-03-20 to 2026-06-26

In [12]:
con.sql(f"""
    SELECT
        COUNT(*)                       AS n_rows,
        COUNT(DISTINCT instrument_id)  AS n_contracts,
        MIN(trading_date)              AS earliest,
        MAX(trading_date)              AS latest
    FROM read_parquet('{MASTER_PATH}')
""").df()

,n_rows,n_contracts,earliest,latest
0,5615794,20252,2023-03-20,2026-06-26


**Missing futures prices**

81,584 of 1,763,210 rows in the test window carry no futures price, around 4.6%.

The gaps are whole trading days, almost all month-ends. These are the days
between a front-month contract's last trading day and the next monthly
listing, which the front-month series of Sec. 4.1.2 does not cover. Without
a futures price, no moneyness can be computed for those days.

In [11]:
con.sql(f"""
SELECT
    COUNT(*)                                          AS n_rows,
    COUNT(futures_price)                              AS n_with_futures,
    COUNT(*) - COUNT(futures_price)                   AS n_missing
FROM read_parquet('{MASTER_PATH}')
WHERE trading_date BETWEEN DATE '2024-09-05' AND DATE '2026-04-24'
""").df()

con.sql(f"""
SELECT trading_date, COUNT(*) AS n_rows
FROM read_parquet('{MASTER_PATH}')
WHERE trading_date BETWEEN DATE '2024-09-05' AND DATE '2026-04-24'
  AND futures_price IS NULL
GROUP BY 1
ORDER BY 1
""").df()

,trading_date,n_rows
0,2024-09-30,5674
1,2024-10-31,5286
2,2024-11-29,4972
3,2024-12-31,4650
4,2025-01-31,4828
5,2025-02-28,4594
6,2025-03-31,4418
7,2025-04-21,4452
8,2025-04-30,4226
9,2025-05-30,4048


Out of $5615794$ rows $450548$ do not have future prices

In [18]:
con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        COUNT(*) - COUNT(futures_price) AS n_missing
    FROM read_parquet('{MASTER_PATH}')
""").df()

,n_rows,n_missing
0,5615794,450548


Out of $5615794$ settlement-days only $26618$ were traded

In [7]:
con.sql(f"""
    SELECT
        COUNT(*)                                   AS n_settlement_days,
        SUM(CASE WHEN has_trades THEN 1 ELSE 0 END) AS n_with_trade
    FROM read_parquet('{MASTER_PATH}')
""").df()

,n_settlement_days,n_with_trade
0,5615794,26618.0


In [28]:
con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{MASTER_PATH}')
    WHERE days_to_expiry = 0
      AND trading_date BETWEEN DATE '2024-09-05' AND DATE '2026-04-24'
""").df()

,n
0,5062


**Moneyness band — upper bound**

Median settlement price falls smoothly with moneyness and shows no natural break. Above 2.0 prices approach the exchange minimum tick of 0.005 and carry no information.

In [30]:

df = con.sql(f"""
SELECT
    ROUND(moneyness_k_over_f, 1) AS moneyness,
    COUNT(*)                     AS n,
    MEDIAN(settlement_price)     AS median_price
FROM read_parquet('{MASTER_PATH}')
WHERE trading_date BETWEEN DATE '2024-09-05' AND DATE '2026-04-24'
  AND days_to_expiry > 0
  AND futures_price IS NOT NULL
  AND instrument_class = 'C'
  AND moneyness_k_over_f BETWEEN 0.3 AND 3.0
GROUP BY 1 ORDER BY 1
""").df()
print(df.to_string())

    moneyness      n  median_price
0         0.3  11345       21.8220
1         0.4  33532       17.1205
2         0.5  53642       11.3200
3         0.6  75541        9.3200
4         0.7  78063        7.7580
5         0.8  73923        6.2730
6         0.9  67172        5.0060
7         1.0  59752        4.0385
8         1.1  47255        3.3320
9         1.2  39422        2.7080
10        1.3  32669        2.1990
11        1.4  29280        1.6810
12        1.5  23946        1.3000
13        1.6  18456        1.1140
14        1.7  16008        0.9190
15        1.8  14846        0.8450
16        1.9  13412        0.7140
17        2.0  13385        0.5310
18        2.1  12089        0.4970
19        2.2  11319        0.3950
20        2.3  10408        0.2900
21        2.4   9564        0.2290
22        2.5   7896        0.1850
23        2.6   6809        0.1490
24        2.7   5872        0.1050
25        2.8   5595        0.1040
26        2.9   4695        0.0650
27        3.0   2306

**Moneyness band — lower bound**
Implied volatility is stable between 0.5 and 1.0, at 0.37 to 0.39. Below 0.5 it rises and becomes erratic: deep in-the-money calls are almost all intrinsic value, so the inversion recovers volatility from a small residual.

In [29]:
df = con.sql(f"""
SELECT
    ROUND(moneyness_k_over_f, 1) AS moneyness,
    COUNT(*)                     AS n,
    MEDIAN(implied_vol)          AS median_iv
FROM read_parquet('{MASTER_PATH}')
WHERE trading_date BETWEEN DATE '2024-09-05' AND DATE '2026-04-24'
  AND days_to_expiry > 0
  AND futures_price IS NOT NULL
  AND instrument_class = 'C'
  AND moneyness_k_over_f BETWEEN 0.1 AND 1.0
GROUP BY 1 ORDER BY 1
""").df()
print(df.to_string())


,moneyness,n,median_iv
0,0.1,1919,0.5506
1,0.2,6961,0.6857
2,0.3,18915,0.5420
3,0.4,33532,0.4553
4,0.5,53642,0.3860
5,0.6,75541,0.3708
6,0.7,78063,0.3717
7,0.8,73923,0.3783
8,0.9,67172,0.3869
9,1.0,31716,0.3944


**Strike coverage by time to expiry**

Within the band, short-dated expiries carry at least 46 strikes. Sparse cases lie beyond one year, where the ladder is not yet fully listed.

In [ ]:
df = con.sql(f"""
SELECT
    days_to_expiry,
    COUNT(DISTINCT strike_price) AS n_strikes
FROM read_parquet('{MASTER_PATH}')
WHERE trading_date BETWEEN DATE '2024-09-05' AND DATE '2026-04-24'
  AND days_to_expiry > 0
  AND futures_price IS NOT NULL
  AND instrument_class = 'C'
  AND moneyness_k_over_f BETWEEN 0.5 AND 2.0
GROUP BY trading_date, expiration_date, days_to_expiry
""").df()
df["bucket"] = pd.cut(df.days_to_expiry, [0, 30, 60, 90, 180, 365, 10000])
print(df.groupby("bucket", observed=True)["n_strikes"].describe())

In [31]:
# how many unique expiries are there in the test split?
con.sql(f"""
SELECT DISTINCT expiration_date
FROM read_parquet('{MASTER_PATH}')
WHERE expiration_date BETWEEN DATE '2024-09-05' AND DATE '2026-04-24'
ORDER BY 1
""").df()

,expiration_date
0,2024-09-26
1,2024-10-25
2,2024-11-26
3,2024-12-27
4,2025-01-27
5,2025-02-24
6,2025-03-27
7,2025-04-25
8,2025-05-27
9,2025-06-26
